# 02. Обучение ConvLSTM v2

**Цель:** обучить ConvLSTM на подготовленных в 01 данных.

**Гиперпараметры (из v2 финальной версии):**
- LR = 1e-4
- Epochs = 80 (с early stopping patience 20)
- Batch size = 8
- Optimizer = Adam
- Scheduler = ReduceLROnPlateau (factor=0.5, patience=10)
- Loss = masked MSE (учитывает NaN в target)

**Acceptance:** best val loss < 0.06, best epoch ~ 15-25 (по опыту v2).

In [ ]:
# Environment auto-detection
# Работает на Colab VM, локальном Jupyter, и Colab UI с local runtime
import os
from pathlib import Path

IN_COLAB_VM = (
    'COLAB_RELEASE_TAG' in os.environ or
    'COLAB_GPU' in os.environ
)

if IN_COLAB_VM:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE_DIR = Path('/content/drive/MyDrive/AI4Arctic')
    env_label = 'Colab VM'
else:
    BASE_DIR = Path(os.environ.get(
        'AI4ARCTIC_HOME',
        Path.home() / 'Ai4Arctic'
    ))
    env_label = 'Local runtime'

print(f"Environment: {env_label}")
print(f"BASE_DIR: {BASE_DIR}")
assert BASE_DIR.exists(), f"BASE_DIR не найден: {BASE_DIR}"

import sys
sys.path.insert(0, str(BASE_DIR))

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# Device: CUDA → MPS (Apple Silicon) → CPU
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

# Пути
DATA_DIR = BASE_DIR / 'data'
MODELS_DIR = BASE_DIR / 'models'
RESULTS_DIR = BASE_DIR / 'results'
FIGURES_DIR = RESULTS_DIR / 'figures'
METRICS_DIR = RESULTS_DIR / 'metrics'

for d in [MODELS_DIR, FIGURES_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

## 1. Загружаем подготовленные данные

In [ ]:
from torch.utils.data import DataLoader
from src.data import TileDataset
from src.model import ConvLSTMNet
from src.metrics import masked_mse, masked_metrics

pairs = np.load(DATA_DIR / 'train_val_pairs.npz')
norm = np.load(DATA_DIR / 'normalization_stats.npz')

Xtr, ytr, mtr = pairs['Xtr'], pairs['ytr'], pairs['mtr']
Xv, yv, mv = pairs['Xv'], pairs['yv'], pairs['mv']

print(f"Train: {Xtr.shape[0]} тайлов, Val: {Xv.shape[0]}")
print(f"y_mean={float(norm['y_mean']):.4f}, y_std={float(norm['y_std']):.4f}")

In [ ]:
BATCH = 8

train_loader = DataLoader(TileDataset(Xtr, ytr, mtr), batch_size=BATCH, shuffle=True)
val_loader = DataLoader(TileDataset(Xv, yv, mv), batch_size=BATCH, shuffle=False)

print(f"Train batches: {len(train_loader)}, val batches: {len(val_loader)}")

## 2. Инициализация модели

In [ ]:
import random

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if device == 'cuda':
    torch.cuda.manual_seed_all(SEED)

model = ConvLSTMNet(in_channels=20, hidden_ch=32, dropout=0.2).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Параметров: {n_params:,}")

## 3. Цикл обучения

In [ ]:
LR = 1e-4
EPOCHS = 80
MAX_NO_IMPROVE = 20

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=10
)

history = {'train_loss': [], 'val_loss': [], 'val_mae': [], 'val_r2': []}
best_val = float('inf')
best_epoch = -1
best_state = None
patience_no_improve = 0

print(f"Обучение: {EPOCHS} эпох, lr={LR}, patience={MAX_NO_IMPROVE}\n")
for epoch in range(EPOCHS):
    model.train()
    tr_loss = 0.0; n = 0
    for X, y, m in train_loader:
        X, y, m = X.to(device), y.to(device), m.to(device)
        optimizer.zero_grad()
        pred = model(X)
        loss = masked_mse(pred, y, m)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        tr_loss += loss.item() * X.size(0); n += X.size(0)
    tr_loss /= n

    model.eval()
    val_loss = 0.0; n = 0
    vp, vt, vm = [], [], []
    with torch.no_grad():
        for X, y, m in val_loader:
            X, y, m = X.to(device), y.to(device), m.to(device)
            pred = model(X)
            loss = masked_mse(pred, y, m)
            val_loss += loss.item() * X.size(0); n += X.size(0)
            vp.append(pred.cpu()); vt.append(y.cpu()); vm.append(m.cpu())
    val_loss /= n
    mae, rmse, r2 = masked_metrics(torch.cat(vp), torch.cat(vt), torch.cat(vm))

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(val_loss)
    history['val_mae'].append(mae)
    history['val_r2'].append(r2)
    scheduler.step(val_loss)

    is_best = val_loss < best_val
    if is_best:
        best_val = val_loss
        best_epoch = epoch + 1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        patience_no_improve = 0
    else:
        patience_no_improve += 1

    star = ' *' if is_best else ''
    lr_cur = optimizer.param_groups[0]['lr']
    print(f"epoch {epoch+1:3d}: tr {tr_loss:.4f}  val {val_loss:.4f}  "
          f"MAE {mae:.3f}  R2 {r2:+.3f}  lr {lr_cur:.0e}{star}")

    if patience_no_improve >= MAX_NO_IMPROVE:
        print(f"\nРанний стоп — нет улучшений {MAX_NO_IMPROVE} эпох подряд")
        break

print(f"\nЛучшая эпоха: {best_epoch}, val_loss={best_val:.4f}")

## 4. Сохранение чекпоинта

In [ ]:
CHECKPOINT_PATH = MODELS_DIR / 'convlstm_ttop_rk_v2.pt'

torch.save({
    'model_state': best_state,
    'y_mean': float(norm['y_mean']),
    'y_std': float(norm['y_std']),
    'x_mean': norm['x_mean'],
    'x_std': norm['x_std'],
    'rk_map': norm['rk_map'],
    'best_epoch': best_epoch,
    'history': history,
    'description': 'ConvLSTM v2, rk-landcover target, lr=1e-4, 80 epochs',
}, CHECKPOINT_PATH)
print(f"Сохранено: {CHECKPOINT_PATH}")
print(f"Размер: {CHECKPOINT_PATH.stat().st_size/1e6:.2f} МБ")

## 5. Графики обучения

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['val_loss'], label='val')
axes[0].axvline(best_epoch - 1, color='r', ls='--', alpha=0.5, label=f'best ep {best_epoch}')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss (masked MSE)')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_title('Loss')

axes[1].plot(history['val_mae'])
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val MAE')
axes[1].grid(alpha=0.3); axes[1].set_title('Val MAE')

axes[2].plot(history['val_r2'])
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Val R2')
axes[2].grid(alpha=0.3); axes[2].set_title('Val R2')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'training_curves_v2.png', dpi=150, bbox_inches='tight')
plt.show()